# Importation des modules

In [1]:
import pandas as pd
import numpy as np
import os
import statsmodels

In [2]:
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]


# Chargement des données
diviser les dépenses de santé par le PIB pour les avoir en %  
rajouter les données de dépenses de santé pour 1994 - 2004 en regardant sur les données de l'ocde  
rajouter la base mortalite  
faire le tri dans les pays de la bdd nb de médecins  


In [3]:
def load_data() :
    #PIB, pop_65, pop_tot, densite_medicale, out_of_pocket, mortalite :  #à remplacer par load_data() quand ce sera codé

    depenses_sante_vol = pd.read_excel(os.path.join("data", "Depenses_sante_en_volume.xlsx"))
    depenses_sante_PIB = pd.read_excel(os.path.join("data", "Depenses_Sante_PIB.xlsx"))
    pib = pd.read_excel(os.path.join("data", "PIB.xlsx"))
    pib_par_habitant = pd.read_excel(os.path.join("data", "PIB_par_habitant.xlsx"))
    pop_tot = pd.read_excel(os.path.join("data", "Pop_tot.xlsx"))
    pop65 = pd.read_excel(os.path.join("data", "Pop+65ans.xlsx"))
    part_pop65 = pd.read_excel(os.path.join("data", "part_pop_plus_65.xlsx"))
    pop_par_age = pd.read_excel(os.path.join("data", "Population_par_age.xlsx"))
    
    return depenses_sante_vol, depenses_sante_PIB, pib, pib_par_habitant, pop_tot, pop65, part_pop65, pop_par_age

depenses_sante_vol, depenses_sante_PIB, pib, pib_par_habitant, pop_tot, pop65, part_pop65, pop_par_age = load_data()

In [11]:
pop_par_age

,TIME,1994,1994.1,1994.2,1994.3,1994.4,1994.5,1994.6,1994.7,1994.8,...,2024.25,2024.26,2024.27,2024.28,2024.29,2024.30,2024.31,2024.32,2024.33,2024.34
0,AGE (Libellés),65 ans,66 ans,67 ans,68 ans,69 ans,70 ans,71 ans,72 ans,73 ans,...,90 ans,91 ans,92 ans,93 ans,94 ans,95 ans,96 ans,97 ans,98 ans,99 ans
1,Belgique,106608,103315,103263,103012,100302,98046,93460,92333,88971,...,29092,25482,20922,16938,12191,8939,6442,4518,3065,2092
2,Tchéquie,100610,96572,96927,95181,94085,93604,89762,84671,73522,...,16672,13911,10864,8469,5772,4086,2643,1846,1179,705
3,Danemark,47060,45492,45946,45359,45459,44346,42023,43339,43267,...,10502,8568,6802,5408,4154,3262,2281,1690,1146,836
4,Allemagne,857950,793902,776619,766438,706274,687906,710515,729129,702383,...,184728,152699,126570,105532,78078,58232,40271,27964,18726,11717
5,Estonie,16497,15771,14498,13169,12606,12119,10867,9890,7757,...,2921,2621,2143,1586,1209,921,610,393,255,184
6,Irlande,25383,25065,25668,25717,25378,24210,23665,22793,22433,...,7102,6364,4861,4175,2949,2210,1786,1239,938,677
7,Grèce,130228,115008,122356,114042,91756,76250,83711,72273,72462,...,:,:,:,:,:,:,:,:,:,:
8,Espagne,411242,385161,387872,369230,360060,346265,339821,311341,290270,...,142898,122334,96780,79709,59706,46107,32357,23636,16310,11157
9,France,569035,552452,548662,539615,519351,510816,502433,507204,501910,...,196799,177323,148566,120839,91884,72440,53392,40085,28314,19114


# Construction du Time-to-Death

In [5]:
def time_to_death(mortalite, pop) :
    # pour pop, à voir si on prend pop_tot, pop_65 ou une population plus âgée encore
    # ou faire une somme pondérée des mortalités pour les 65-70, 70-75, etc
    # fait par chatgpt, à revoir en fonction de la structure des données
    df = mortalite.merge(
        pop_tot,
        on=["country", "year", "age"],
        how="inner"
    )

    df["weighted_mortality"] = df["mortality_rate"] * df["population"]
    ttd = (
        df.groupby(["country", "year"])["weighted_mortality"]
          .sum()
          .reset_index(name="TTD")
    )
    return ttd

#pop = pop_65   # ou pop_80, ou pop_tot...
#ttd = time_to_death(mortalite, pop)

# Mise en forme du panel

In [7]:
def panel(depenses_sante, pib, pop_65, pop_tot, densite_medicale, oop, ttd) :
    #construction du panel contenant toutes les données dont on a besoin dans un seul panel
    panel = (
        depenses_sante.merge(pop_65, on=["country", "year"])
           .merge(pib, on=["country", "year"])
           .merge(densite_medicale, on=["country", "year"])
           .merge(oop, on=["country", "year"])
           .merge(ttd, on=["country", "year"])
    )

    panel = panel.sort_values(["country", "year"])
    panel.set_index(["country", "year"], inplace=True)
    return panel

#panel = panel(depenses_sante, pib, pop_65, pop_tot, densite_medicale, oop, ttd)

# Régression et GMM